# Question 22 - 6. Zigzag Conversion

The string `"PAYPALISHIRING"` is written in a zigzag pattern on a given number of rows like this:

    P   A   H   N
    A P L S I I G
    Y   I   R

And then read line by line: `"PAHNAPLSIIGYIR"`

Write the code that will take a string and make this conversion given a number of rows:

    string convert(string s, int numRows);

**Example 1:**

    Input: s = "PAYPALISHIRING", numRows = 3
    Output: "PAHNAPLSIIGYIR"

**Example 2:**

    Input: s = "PAYPALISHIRING", numRows = 4
    Output: "PINALSIGYAHRPI"
    Explanation:
    P     I    N
    A   L S  I G
    Y A   H R
    P     I

**Example 3:**

    Input: s = "A", numRows = 1
    Output: "A"

**Constraints:**

- `1 <= s.length <= 1000`
- `s` consists of English letters (lower-case and upper-case), `','` and `'.'`.
- `1 <= numRows <= 1000`

---
## 🎯 Interview Walkthrough: Brute Force → Optimized

**Pattern:** Simulation with one bucket per row (row index bounces top ↔ bottom)

📘 **Revise first:** [Pattern H · simulate carefully](./0-concepts-array-and-string.ipynb) · [Part 3.11 · 2D lists](./0-concepts-array-and-string.ipynb) · [Part 5.5 · join](./0-concepts-array-and-string.ipynb) in the concepts notebook

| Approach | Time | Space |
|---|---|---|
| 1️⃣ Brute force (draw the full 2D grid) | O(numRows · n) | O(numRows · n) |
| 2️⃣ Optimized (row buckets) | O(n) | O(n) |
| ➕ Alternative (jump by cycle length) | O(n) | O(1) extra |

> 🧠 = plain-English intuition (how you'd explain it to a friend) · 🗣️ = **interview script**: what to say out loud at each step.

### 🧠 The problem in plain English

Write the letters in a **zigzag**: down one row at a time, then diagonally back up to the top, then down again… Then read the result **row by row**, top to bottom.

```
s = "PAYPALISHIRING", numRows = 3

P   A   H   N        row 0
A P L S I I G        row 1
Y   I   R            row 2

read rows: "PAHN" + "APLSIIG" + "YIR" = "PAHNAPLSIIGYIR"
```

**Everyday picture: a ball bouncing between a floor and a ceiling.** Picture `numRows` shelves. Drop letters one at a time onto shelves 0, 1, 2 (going down), and when you hit the bottom shelf, bounce back up: 1, 0. Then down again: 1, 2, and so on.

**The key realisation:** the final answer only depends on **which shelf each letter lands on**. The sideways position (columns) only matters for drawing the picture. So we can throw the picture away and keep **one list per shelf**.

### Step 0 · Clarify before coding

🗣️ *"Letters go down the rows, then diagonally up, repeating, and I read the rows top to bottom. Only the row of each character affects the output."*

| Question to ask | Answer | Why it matters |
|---|---|---|
| `numRows == 1`? | No zigzag; return `s` as it is. | **Must special-case it**, or the bounce logic gets stuck. |
| `numRows ≥ len(s)`? | Each letter gets its own row → output equals `s`. | A quick early return. |
| Characters? | Letters, `,` and `.`. | Nothing special. |

### Step 1 · Brute force: the "obvious" way

**Plain-English idea:** literally draw it. Make a grid with `numRows` rows and enough columns, place each letter as you zig and zag, then read the grid row by row, skipping the blanks.

🗣️ *"The most literal solution builds the 2D picture and reads it row by row. It's correct, but the grid is mostly empty: numRows × n cells for n letters. With 1000 rows and 1000 letters that's a million cells. And the column positions never affect the answer."*

In [1]:
class SolutionBrute:
    def convert(self, s: str, numRows: int) -> str:
        if numRows == 1 or numRows >= len(s):
            return s
        grid = [[""] * len(s) for _ in range(numRows)]
        row, col, going_down = 0, 0, True
        for ch in s:
            grid[row][col] = ch
            if going_down:
                if row == numRows - 1:      # hit the bottom: start going up diagonally
                    going_down = False
                    row -= 1
                    col += 1
                else:
                    row += 1
            else:
                if row == 0:                # hit the top: start going down
                    going_down = True
                    row += 1
                else:
                    row -= 1
                    col += 1
        return "".join(ch for r in grid for ch in r if ch)

### Step 2 · Optimize: the "aha" moment

> 💡 **Aha:** throw away the columns. Keep **one bucket (list) per row**, walk the string once, drop each letter into the bucket for the current row, and **bounce the row number** between the top and the bottom. Finally, glue the buckets together.

The bounce is just a `step` that is `+1` (going down) or `−1` (going up):
- At row 0 → `step = +1`
- At the last row → `step = −1`

🗣️ *"Only the row of each character matters, so I keep one list per row. I walk the string with a row pointer, append each character to its row, and flip direction when I reach the top or bottom row. At the end I join the rows in order. That's O(n) time and O(n) space for the buckets. I special-case one row, because the direction would never flip."*

#### 🤔 Why this approach, and why not the others?

| Option | Verdict | Plain-English reason |
|---|---|---|
| Full 2D grid | ❌ | Mostly empty space, plus column bookkeeping that adds bugs. |
| **Row buckets + bouncing row index** | ✅ **best to code live** | Direct, easy to explain, no wasted cells. |
| Row strings with `+=` | ⚠️ | Usually OK in CPython, but lists + `join` are guaranteed linear. |
| **Cycle formula** | ✅ alternative | The pattern repeats every `2·(numRows − 1)` letters, so you can jump straight to each row's letters with arithmetic. O(1) extra memory, but more index maths to get right under pressure. |

**The cycle idea in plain words:** one full "down and up" uses `cycle = 2·(numRows − 1)` letters. Row `r` gets the letter at `r` in each cycle, and every **middle** row also gets one from the upward diagonal, at `cycle − r` in each cycle.

In [2]:
class Solution:
    def convert(self, s: str, numRows: int) -> str:
        if numRows == 1 or numRows >= len(s):
            return s
        rows = [[] for _ in range(numRows)]     # one bucket per row
        row, step = 0, 1
        for ch in s:
            rows[row].append(ch)
            if row == 0:
                step = 1                        # bounce off the top
            elif row == numRows - 1:
                step = -1                       # bounce off the bottom
            row += step
        return "".join("".join(r) for r in rows)


class SolutionCycle:
    def convert(self, s: str, numRows: int) -> str:
        if numRows == 1 or numRows >= len(s):
            return s
        n, cycle = len(s), 2 * (numRows - 1)
        out = []
        for r in range(numRows):
            for base in range(0, n, cycle):
                if base + r < n:
                    out.append(s[base + r])                      # going-down letter
                if 0 < r < numRows - 1 and base + cycle - r < n:
                    out.append(s[base + cycle - r])              # going-up (diagonal) letter
        return "".join(out)

### Step 3 · Toy example walkthrough (tell it like a story)

`s = "PAYPALISHIRING"`, `numRows = 3`. The row bounces 0, 1, 2, 1, 0, 1, 2, 1, 0, …

| letter | P | A | Y | P | A | L | I | S | H | I | R | I | N | G |
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|
| row | 0 | 1 | 2 | 1 | 0 | 1 | 2 | 1 | 0 | 1 | 2 | 1 | 0 | 1 |

Buckets:
- row 0: `P A H N`
- row 1: `A P L S I I G`
- row 2: `Y I R`

Joined: `PAHN` + `APLSIIG` + `YIR` = **`PAHNAPLSIIGYIR`** ✅

**Narrated:** "P goes on shelf 0, A on shelf 1, Y on shelf 2, the bottom, so I bounce. P goes back to shelf 1, A to shelf 0, the top, so I bounce again. Then L on 1, I on 2, and so on. At the end I read shelf 0, then shelf 1, then shelf 2."

**Cycle view:** cycle = 2·(3 − 1) = 4. Row 0 takes positions 0, 4, 8, 12 → P, A, H, N ✅.

In [3]:
cases = [
    ("PAYPALISHIRING", 3, "PAHNAPLSIIGYIR"),
    ("PAYPALISHIRING", 4, "PINALSIGYAHRPI"),
    ("A", 1, "A"),
    ("AB", 1, "AB"),
    ("ABC", 5, "ABC"),          # more rows than letters
    ("ABCD", 2, "ACBD"),
]
for s, rows, expected in cases:
    for Impl in (SolutionBrute, Solution, SolutionCycle):
        assert Impl().convert(s, rows) == expected, Impl.__name__
    print(s, rows, "->", expected)

import random, string
for _ in range(500):
    s = "".join(random.choices(string.ascii_letters, k=random.randint(1, 20)))
    r = random.randint(1, 7)
    assert Solution().convert(s, r) == SolutionBrute().convert(s, r) == SolutionCycle().convert(s, r)
print("All tests passed ✅")

PAYPALISHIRING 3 -> PAHNAPLSIIGYIR
PAYPALISHIRING 4 -> PINALSIGYAHRPI
A 1 -> A
AB 1 -> AB
ABC 5 -> ABC
ABCD 2 -> ACBD
All tests passed ✅


### Step 4 · Complexity in plain words, and wrap up

| | Time | Extra memory | In plain words |
|---|---|---|---|
| 2D grid | O(numRows · n) | O(numRows · n) | Builds and scans a mostly empty picture. |
| **Row buckets** | **O(n)** | **O(n)** | Each letter goes into one bucket once and is copied out once. |
| Cycle formula | O(n) | O(1) besides the output | Computes each letter's place by arithmetic. |

**Why O(n) time?** Each letter is appended to exactly one bucket, and the final join copies each letter once.

**Why O(n) memory?** The buckets together hold all n letters.

**Edge cases to mention:** one row (return as is), at least as many rows as letters (return as is), two rows (letters alternate).

---

#### 🗣️ Full interview script (about 45 seconds)

*"Only the row each character lands in affects the output; the columns are just for drawing. So I keep one list per row and walk the string with a row index that bounces: it goes down until the last row, then up until row 0, and so on. Each character is appended to its row's list, and at the end I join the rows top to bottom.*

*That's O(n) time and O(n) space. I special-case one row, and more rows than characters, by returning the string unchanged. If asked for O(1) extra space, I'd use the cycle length 2·(rows − 1) to jump directly to each row's characters."*